In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
from pathlib import Path

In [7]:
# Remove empty .log files from dir
linear_script_dir = Path('/home/mila/s/shawn.whitfield/projects/AMPLIFY-private/interpretability/project/scripts/linear_probing')
for f in linear_script_dir.glob('*.log'):
    f.unlink()

In [8]:
# Remove error logs from dir
def remove_logs(log_dir:str):
    log_path = Path(log_dir)
    for f in log_path.glob("*.err"):
        f.unlink()
    for f in log_path.glob("*.out"):
        f.unlink()
remove_logs('/home/mila/s/shawn.whitfield/scratch/logs/linear_probing/')

In [9]:
dataset_config_dir = Path('/home/mila/s/shawn.whitfield/projects/AMPLIFY-private/interpretability/project/configs/datamodule/dataset')
for file in dataset_config_dir.iterdir():
    print(file)

/home/mila/s/shawn.whitfield/projects/AMPLIFY-private/interpretability/project/configs/datamodule/dataset/interpro_conserved_site_split1.yaml
/home/mila/s/shawn.whitfield/projects/AMPLIFY-private/interpretability/project/configs/datamodule/dataset/uniprot_membrane_pass_512_cutoff_split2.yaml
/home/mila/s/shawn.whitfield/projects/AMPLIFY-private/interpretability/project/configs/datamodule/dataset/prot_param_512_cutoff_split2.yaml
/home/mila/s/shawn.whitfield/projects/AMPLIFY-private/interpretability/project/configs/datamodule/dataset/uniprot_topology_512_cutoff.yaml
/home/mila/s/shawn.whitfield/projects/AMPLIFY-private/interpretability/project/configs/datamodule/dataset/uniprot_phosphorylation_512_cutoff_split2.yaml
/home/mila/s/shawn.whitfield/projects/AMPLIFY-private/interpretability/project/configs/datamodule/dataset/interpro_domain_split2.yaml
/home/mila/s/shawn.whitfield/projects/AMPLIFY-private/interpretability/project/configs/datamodule/dataset/uniprot_domains_split2.yaml
/home/m

In [ ]:
from pathlib import Path
import yaml
dataset_config_dir = Path('/home/mila/s/shawn.whitfield/projects/AMPLIFY-private/interpretability/project/configs/datamodule/dataset')

for file_path in dataset_config_dir.glob('*.yaml'): # Use glob to ensure we only hit yaml files
    if 'biomap' in file_path.name:
        continue
        
    try:
        # 1. Read the original data once
        with open(file_path, 'r') as f:
            config_dict = yaml.safe_load(f)

        dataset_name = config_dict.get('dataset_name', 'default')

        # 2. Create separate files for each split
        for i in range(1, 3):
            # Create a copy so we don't mutate the original dict for the next split
            new_config = config_dict.copy()
            
            # Logic for the new path
            # Assuming 'data_path' is what needs the suffix
            original_path = new_config.get('data_path', '')
            new_path = original_path.replace('.parquet', f'_split{i}.parquet')
            
            new_config['data_path'] = new_path
            new_config['dataset_name'] = f"{dataset_name}_split{i}"

            # 3. Write to a NEW filename to avoid overwriting the source
            new_filename = file_path.parent / f"{file_path.stem}_split{i}.yaml"
            with open(new_filename, 'w') as f:
                yaml.dump(new_config, f, default_flow_style=False)
                
    except Exception as e:
        print(f"Error processing {file_path}: {e}")

In [ ]:
from project.algorithms.networks.protein_language_model import ProteinLanguageModel
from project.utils.strs import plms

model_name = plms['mila_amplify_120m_100000']['full_name']

plm = ProteinLanguageModel(model_name=model_name, layer_to_use=None)
plm

In [ ]:
for fn in plms.values():
    print(fn['full_name'])

In [ ]:
sequences = ['MSDITNNSDGGG', 'MSDITNAAAANSDGGG']

In [ ]:
plm(sequences)

In [ ]:
import numpy as np

In [ ]:
rng = np.random.default_rng(seed=5)
np.zeros((5,5))[]

In [ ]:
rng.choice(10,2, replace=False)

In [ ]:
from Bio import SeqIO

# The file names are long, so we'll use variables to store them
fasta_file_name = "interpro_wwwapi_protein_UniProt_entry_InterPro_proteome_uniprot_UP000005640_.fasta"
accession_file_name = "interpro_wwwapi_entry_InterPro_proteote_uniprot_UP000005640_.accession"

# --- Reading the FASTA file with Biopython ---
print(f"--- Reading {fasta_file_name} with Biopython ---")
sequences = {}
try:
    # SeqIO.parse returns an iterator that yields SeqRecord objects
    for record in SeqIO.parse(fasta_file_name, "fasta"):
        # Store the SeqRecord objects using the ID (e.g., A0A024QZ33) as the key
        sequences[record.id] = record
        
    print(f"Total number of protein sequences parsed: {len(sequences)}")
    
    # Example: Accessing data for the first sequence
    first_id = next(iter(sequences))
    first_record = sequences[first_id]
    
    print(f"\nExample Sequence ID: {first_record.id}")
    print(f"Description (Full Header): {first_record.description}")
    print(f"Length of Sequence: {len(first_record.seq)}")
    print(f"First 50 amino acids: {first_record.seq[:50]}...")
    
except FileNotFoundError:
    print(f"Error: The file '{fasta_file_name}' was not found.")
except Exception as e:
    print(f"An error occurred during Biopython parsing: {e}")

# --- Reading the Accession file (basic Python is fine here) ---
print(f"\n--- Reading {accession_file_name} ---")
try:
    with open(accession_file_name, 'r') as f:
        accession_codes = [line.strip() for line in f if line.strip()]
        
    print(f"Total accession codes found: {len(accession_codes)}")
    
except FileNotFoundError:
    print(f"Error: The file '{accession_file_name}' was not found.")

In [ ]:
sequences

human ID: ID of the human protein
GOs: Gene Ontology terms associated with the protein
rep ID: ID of the representative protein if the human protein is clustered in the Foldseek clustering step
cluFlag 1: clustered in AFDB50, 2: clustered in AFDB clusters, 3: removed (fragments in Foldseek clusters), 4: removed (singletons in Foldseek clusters)
LCA taxonomy ID: LCA taxonomy ID of the cluster where the human protein is assigned

In [ ]:
from pathlib import Path
import polars as pl
import torch

from project.utils.functions import load_config
from project.utils.strs import plms

In [ ]:
from pathlib import Path
import yaml
dataset_config_dir = Path('/home/mila/s/shawn.whitfield/projects/AMPLIFY-private/interpretability/project/configs/datamodule/dataset')
for file in dataset_config_dir.iterdir():
    try:
        with open(dataset_config_dir / str(file), 'r') as f:
            dataset_config_dict = yaml.safe_load(f)
            data_path = dataset_config_dict['data_path']
            data_path_replaced = data_path.replace('/network/scratch/s/', '/home/mila/s/')
            # data_path_replaced = data_path.replace('/home/mila/s/', '/network/scratch/s/')
            # print(data_path_replaced)
            dataset_config_dict['data_path'] = data_path_replaced
            with open(file, 'w') as f:
                # Use yaml.dump to write the dictionary back to the file
                # The default_flow_style=False makes the output more readable (multi-line style)
                yaml.dump(dataset_config_dict, f, default_flow_style=False)
    except:
        continue

In [ ]:
from pathlib import Path
def remove_logs(log_dir:str):
    log_path = Path(log_dir)
    for f in log_path.glob("*.err"):
        f.unlink()
    for f in log_path.glob("*.out"):
        f.unlink()
# remove_logs('/home/mila/s/shawn.whitfield/scratch/logs/linear_probing/')
remove_logs('/home/mila/s/shawn.whitfield/scratch/logs/neighborhood/')


# def compress_parquets(target_dir:str):
#     from pathlib import Path
#     target_path = Path(target_dir)
#     for f in target_path.glob("*.parquet"):
#         fname = f.name
#         df = pl.read_parquet(f)
#         new_name = str(f) + '.gz'
#         df.write_parquet(new_name, compression='gzip')
#         f.unlink()

# compress_parquets('/home/mila/s/shawn.whitfield/scratch/results/nearest_neighbor')

In [ ]:
df = pl.read_parquet(load_config('interpro_active_site')['data_path'])
df

In [ ]:
TARGET_DATASETS = [
    # "protein properties"
	"prot_param",
    # Dependent largely on linear sequence
	"interpro_conserved_site",
	"interpro_repeat",
    "uniprot_peptide",
    "uniprot_post_translational_modification",
	"biomap_localization_prediction",
    # Secondary structure
    "biomap_ssp_q3",
    "biomap_ssp_q8",
	"uniprot_secondary_structure",
    # Dependent largely on tertiary structure
    "uniprot_functional_sites",
	"interpro_binding_site",
	"biomap_metal_ion_binding",
	"interpro_active_site",
    "interpro_domain",
    "interpro_family",
    # Dependent on cellular context
    "uniprot_topology",
    "GO_cc",
    "GO_mf",
    "GO_bp",
    ]
for dn in TARGET_DATASETS:
    print(dn)
    df = pl.read_parquet(load_config(dn)['data_path'])
    print('train val test')
    print(len(df.filter(pl.col('split')=='train')), len(df.filter(pl.col('split')=='val')), len(df.filter(pl.col('split')=='test')))

In [ ]:
from project.utils.strs import plms

MODEL_NAMES = [
    "chandar-lab/AMPLIFY_120M",
    "chandar-lab/AMPLIFY_350M",
    "chandar-lab/SaAMPLIFY_120M",
    "chandar-lab/SaAMPLIFY_350M",
    "facebook/esm2_t6_8M_UR50D",
    "facebook/esm2_t12_35M_UR50D",
    "facebook/esm2_t30_150M_UR50D",
    "facebook/esm2_t33_650M_UR50D",
]

models_to_hiddens = {v['full_name']: v['num_hiddens'] for v in plms.values()}

MODEL_LAYERS = {model_name: list(range(models_to_hiddens[model_name])) for model_name in MODEL_NAMES
}

TARGET_DATASETS = [
    # "protein properties"
	"prot_param",
    # Dependent largely on linear sequence
	"interpro_conserved_site",
	"interpro_repeat",
    "uniprot_peptide",
    "uniprot_post_translational_modification",
	"biomap_localization_prediction",
    # Secondary structure
    "biomap_ssp_q3",
    "biomap_ssp_q8",
	"uniprot_secondary_structure",
    # Dependent largely on tertiary structure
    "uniprot_functional_sites",
	"interpro_binding_site",
	"biomap_metal_ion_binding",
	"interpro_active_site",
    "interpro_domain",
    "interpro_family",
    # Dependent on cellular context
    "uniprot_topology",
    "GO_cc",
    "GO_mf",
    "GO_bp",
]

required_runs = {}
for mn in MODEL_NAMES:
    print(mn)
    num_layers = len(MODEL_LAYERS[mn])
    required_runs[mn] = num_layers * len(TARGET_DATASETS)
required_runs

In [ ]:
def delete_log_files(dir):
	for file in dir.glob('*.log'):
		try:
			file.unlink()
		except Exception as e:
			print(f"Error deleting {e}")

delete_log_files(Path('/home/mila/s/shawn.whitfield/projects/AMPLIFY-private/interpretability/project/scripts/linear_probing'))

In [ ]:
def get_model_dataset_num_checkpoints(model_dir):
    """ 
    Given a model dir, iterate through and find all checkpoints
    """
    MODEL_NAMES = [
    "chandar-lab/AMPLIFY_120M",
    "chandar-lab/AMPLIFY_350M",
    "chandar-lab/SaAMPLIFY_120M",
    "chandar-lab/SaAMPLIFY_350M",
    "facebook/esm2_t6_8M_UR50D",
    "facebook/esm2_t12_35M_UR50D",
    "facebook/esm2_t30_150M_UR50D",
    "facebook/esm2_t33_650M_UR50D",
    ]
    TARGET_DATASETS = [
    # "protein properties"
	"prot_param",
    # Dependent largely on linear sequence
	"interpro_conserved_site",
	"interpro_repeat",
    "uniprot_peptide",
    "uniprot_post_translational_modification",
	"biomap_localization_prediction",
    # Secondary structure
    "biomap_ssp_q3",
    "biomap_ssp_q8",
	"uniprot_secondary_structure",
    # Dependent largely on tertiary structure
    "uniprot_functional_sites",
	"interpro_binding_site",
	"biomap_metal_ion_binding",
	"interpro_active_site",
    "interpro_domain",
    "interpro_family",
    # Dependent on cellular context
    "uniprot_topology",
    "GO_cc",
    "GO_mf",
    "GO_bp",
    ]


    checkpoints_info = []

    for f in model_dir.iterdir():
        # Parse /home/mila/s/shawn.whitfield/scratch/models/linear_probes/interpro_conserved_site_facebook-esm2_t6_8M_UR50D_layer_1/checkpoints/epoch_028.ckpt
        filename = f.name
        model_name = [mn for mn in MODEL_NAMES if (mn.replace('/','-') in filename)][0]
        dataset = [ds for ds in TARGET_DATASETS if (ds in filename)][0]
        layer_num = int(filename.split('layer_')[-1])

        checkpoints_dir = f / 'checkpoints'
        if not checkpoints_dir.is_dir():
            continue
        for checkpoint_file in checkpoints_dir.iterdir():
            if 'last' not in str(checkpoint_file):
                filename = checkpoint_file.name
                if 'epoch_' in filename and '.ckpt' in filename:
                    # parse the checkpoint number
                    # e.g., 'epoch_028.ckpt' -> '028' -> 28
                    try:
                        checkpoint_number = int(filename.split('.ckpt')[0].split('epoch_')[1])
                    except (ValueError, IndexError):
                        # Skip files that don't match the expected naming pattern
                        continue
                else:
                    continue # Skip other files in the checkpoints directory

                checkpoints_info.append({'model_name': model_name,
                'dataset': dataset,
                'layer_num': layer_num,
                'best_checkpoint': checkpoint_number})

        

In [ ]:
from project.utils.strs import linear_probe_dir

for f in linear_probe_dir.iterdir():
    checkpoints_dir = f / 'checkpoints'
    if not checkpoints_dir.is_dir():
        continue
    for checkpoint_file in checkpoints_dir.iterdir():
        try:
            print(checkpoint_file)
            # Read the file
            checkpoint = torch.load(checkpoint_file, map_location='cpu')
            # 2. Extract the full state dictionary
            checkpoint_state_dict = checkpoint.get('state_dict')
            if checkpoint_state_dict is None:
                print("Warning: 'state_dict' key not found. Skipping.")
                continue
                
            # 3. Filter keys: Keep only those starting with 'linear_probe.'
            # This assumes your old module's state dict *already* used the prefix.
            new_state_dict = {
                k: v 
                for k, v in checkpoint_state_dict.items() 
                if k.startswith('linear_probe')
            }
            
            # 4. Update the checkpoint object with the minimal state dict
            checkpoint["state_dict"] = new_state_dict

            # 5. Overwrite the file with the modified checkpoint
            # Saving the modified 'checkpoint' object back to the file path
            torch.save(checkpoint, checkpoint_file)
        except Exception as e:
            print(e)
            print(checkpoint_file)
            continue

In [ ]:
dataset = 'uniprot_peptide'
df = pl.read_parquet(load_config(dataset)['data_path'])
df[dataset.split('_')[1]].list.explode().unique().to_list()

In [ ]:
probes_dir = Path('/home/mila/s/shawn.whitfield/scratch/models/linear_probes')

In [ ]:
emb = torch.load('/home/mila/s/shawn.whitfield/scratch/data/embeddings/amplify_120m/amplify_120m_layer_0_mean_embeddings.pt', map_location='cpu')
emb.keys()

In [ ]:
emb = torch.load('/home/mila/s/shawn.whitfield/scratch/data/embeddings/esm2_35m/esm2_35m_layer_0_mean_embeddings.pt', map_location='cpu')
emb.keys()

In [ ]:
emb = torch.load('/home/mila/s/shawn.whitfield/scratch/data/embeddings/esm2_650m/esm2_650m_layer_3_mean_embeddings.pt', map_location='cpu')
emb.keys()


In [ ]:
import torch
from pathlib import Path
for model_shorthand in ['esm2_8m', 'esm2_35m', 'amplify_120m', 'samplify_120m', 'esm2_150m']:
    for filename in Path(f'/home/mila/s/shawn.whitfield/scratch/data/embeddings/{model_shorthand}').glob("*mean_embeddings*"):
        try:
            print(f'loading {filename}')
            file = torch.load(filename, map_location='cpu', weights_only=True)
            del file
        except:
            print(f'could not read {filename}')
            continue

In [ ]:
def get_cached_embeddings(sequences, model_shorthand, layer_nums:list[0,1], as_numpy=True):
    from pathlib import Path
    import torch
    embedding_dir = Path('/home/mila/s/shawn.whitfield/scratch/data/embeddings/')
    embedding_model_dir = embedding_dir / model_shorthand

    all_embeddings = {}

    for layer_num in layer_nums:
        filename = embedding_model_dir / f"{model_shorthand}_layer_{layer_num}_mean_embeddings.pt"
        # Load pre-cached embeddings
        embedding_dict = torch.load(filename, map_location='cpu', weights_only=True) # format: {'ids':list: ids, 'sequences':list : seqs, 'embeddings': torch.Tensor(num_seqs, embedding_dim)}
        cached_sequences = embedding_dict['sequences']
        cached_embeddings = embedding_dict['embeddings']
        # Find where the requested sequences are in the list
        sequence_to_index = {seq: i for i, seq in enumerate(cached_sequences)}
        seq_inds = [sequence_to_index[seq] for seq in sequences]
            
        # Get those indices in the cached embeddings
        requested_tensor = cached_embeddings[seq_inds,:]

        if as_numpy:
            all_embeddings[layer_num] = requested_tensor.numpy()
        else:
            all_embeddings[layer_num] = requested_tensor

        if requested_tensor.shape[0] != len(sequences):
            print('mismatch in expected size! sequences may be missing')

    return all_embeddings

get_cached_embeddings('MGLEALVPLAMIVAIFLLLVDLMHRHQRWAARYPPGPLPLPGLGNLLHVDFQNTPYCFDQLRRRFGDVFSLQLAWTPVVVLNGLAAVREAMVTRGEDTADRPPAPIYQVLGFGPRSQGVILSRYGPAWREQRRFSVSTLRNLGLGKKSLEQWVTEEAACLCAAFADQAGRPFRPNGLLDKAVSNVIASLTCGRRFEYDDPRFLRLLDLAQEGLKEESGFLREVLNAVPVLPHIPALAGKVLRFQKAFLTQLDELLTEHRMTWDPAQPPRDLTEAFLAKKEKAKGSPESSFNDENLRIVVGNLFLAGMVTTSTTLAWGLLLMILHLDVQRGRRVSPGCPIVGTHVCPVRVQQEIDDVIGQVRRPEMGDQAHMPCTTAVIHEVQHFGDIVPLGVTHMTSRDIEVQGFRIPKGTTLITNLSSVLKDEAVWKKPFRFHPEHFLDAQGHFVKPEAFLPFSAGRRACLGEPLARMELFLFFTSLLQHFSFSVAAGQPRPSHSRVVSFLVTPSPYELCAVPR', 
'esm2_35m', layer_nums=[0,5,11])

In [ ]:
filename = embedding_model_dir / f"{model_shorthand}_layer_{layer_num}_mean_embeddings.pt"

In [ ]:
from pathlib import Path
import torch
import torch.nn.functional as F

data_dir = Path('/home/mila/s/shawn.whitfield/scratch/data')
embedding_dir = data_dir / 'embeddings'
attention_dir = data_dir / 'attentions'
model_shorthands = ['esm2_8m', ] # , 'esm2_150m' 'amplify_120m', 'samplify_120m'

first_n = 10

for model_shorthand in model_shorthands:
    print(model_shorthand)
    # Get the dirs
    for layer_dir in (embedding_dir / model_shorthand).glob(f'*layer_*'):

        print(layer_dir)

        mean_embeddings = []
        embeddings = []
        ids = []
        sequences = []
        max_len = 0

        for filename in list(layer_dir.glob("*batch*"))[:first_n]:
            save_name = str(filename).split('_batch')[0]
            print(filename)

            embedding_dict = torch.load(filename)
            if 'mean_embeddings' in str(filename):
                mean_embeddings.extend(embedding_dict['embeddings'].unsqueeze(0))
            else:
                ids.extend(embedding_dict['ids'])
                sequences.extend(embedding_dict['sequences'])
                embeddings.extend(embedding_dict['embeddings'].unsqueeze(0))
                max_len = max(max_len, embedding_dict['embeddings'].unsqueeze(0).shape[1])
    
        # Pad embeddings to max and concatenate
        padded_embeddings = []
        # Pad dimension 1
        for tensor in embeddings:
            to_pad = max_len - tensor.shape[1]
            padded_tensor = F.pad(tensor, (0,0,0,max_len - tensor.shape[1]), 'constant', 0.0)
            padded_embeddings.append(padded_tensor)

        all_embeddings = torch.cat(padded_embeddings, dim=0)
        all_mean_embeddings = torch.cat(mean_embeddings, dim=0)

        # torch.save({
        #     'ids': ids,
        #     'sequences': sequences,
        #     'embeddings': all_embeddings
        # }, 
        # layer_dir / f"all_embeddings.pt")

        # torch.save({
        #     'ids': ids,
        #     'sequences': sequences,
        #     'embeddings': all_mean_embeddings
        # }, 
        # layer_dir / f"all_mean_embeddings.pt")

        # del embedding_dict
        # del all_embeddings
        # del mean_embeddings
        # del padded_embeddings

    for layer_dir in sorted((attention_dir / model_shorthand).glob(f'*layer_*')):

        ids = []
        sequences = []
        attentions = []
        max_len = 0

        for filename in list(layer_dir.glob("*batch*"))[:first_n]:
            save_name = str(layer_dir).split('_batch')[0]

            attention_dict = torch.load(filename)
            ids.extend(attention_dict['ids'])
            sequences.extend(attention_dict['sequences'])
            attentions.extend(attention_dict['attentions'].unsqueeze(0))
            max_len = max(max_len, attention_dict['attentions'].unsqueeze(0).shape[1])

        padded_attentions = []
        # Pad dimensions 1 and 2
        for tensor in attentions:
            to_pad = max_len - tensor.shape[-1]
            padded_tensor = F.pad(tensor, (0,to_pad,0,to_pad), 'constant', 0.0)
            padded_attentions.append(padded_tensor)

        all_attentions = torch.cat(padded_attentions)

        save_path = layer_dir / f"all_attentions.pt"
        print(save_path)

        # torch.save({
        #     'ids': ids,
        #     'sequences': sequences,
        #     'embeddings': all_attentions
        # }, 
        # save_path)

        # del attention_dict
        # del all_attentions
        # del padded_attentions

In [ ]:
attentions[0].shape

In [ ]:
embedding_dict

In [ ]:
all_attentions

In [ ]:
sorted((attention_dir / model_shorthand).glob(f'*layer_*'))

# Imports

In [ ]:
# Get and store protein language model embeddings for a set of sequences
import torch
import pickle
from tqdm import tqdm # For a progress bar
from pathlib import Path
import polars as po
from project.utils.functions import (set_device, 
    check_correlation, 
    read_fasta, 
    one_hot_polars_column, 
    go_to_readable, 
    load_godag)
from project.utils.strs import PROTEIN_LENGTH_CUTOFF, plms
DEVICE = set_device()
from project.algorithms.networks.protein_language_model import ProteinLanguageModel
from project.datamodules.datamodule import PolarsDataset, PLMDataModule
from torch.utils.data import DataLoader

In [ ]:
# embedding_filepath = Path('/home/mila/s/shawn.whitfield/scratch/data/embeddings/facebook_esm2_t6_8M_UR50D/h_sapiens_proteome_facebook_esm2_t6_8M_UR50D_all_embeddings.pkl')
# with open(embedding_filepath, 'rb') as f:
#     data = pickle.load(f)

/home/mila/s/shawn.whitfield/scratch/data/embeddings/facebook_esm2_t6_8M_UR50D/h_sapiens_proteome_facebook_esm2_t6_8M_UR50D_l1_embeddings.pkl

In [ ]:
def load_embeddings(model_name:str, layer_num:int, dataset='h_sapiens_proteome'):
    """ 
    Load hidden representations of human proteome proteins
    """
    from pathlib import Path
    import pickle

    embedding_path = Path(f'/home/mila/s/shawn.whitfield/scratch/data/embeddings/{model_name.replace("/", "_")}/{dataset}_{model_name.replace("/", "_")}_l{layer_num}_embeddings.pkl')

    with open(embedding_path, 'rb') as f:
        data = pickle.load(f)
    
    return data

In [ ]:
plms

In [ ]:
embeddings = load_embeddings(plms['amplify_120m'], layer_num=1) # 32 Gb mem request can open without crashing

In [ ]:
embeddings

In [ ]:
embeddings['hidden_embeddings'].shape

In [ ]:
embeddings['hidden_embeddings']

In [ ]:
'/home/mila/s/shawn.whitfield/scratch/data/embeddings/facebook_esm2_t6_8M_UR50D/h_sapiens_proteome_facebook_esm2_t6_8M_UR50D_l0_embeddings.pkl'

In [ ]:
godag = load_godag()

In [ ]:
df = po.read_parquet('/home/mila/s/shawn.whitfield/scratch/data/tasks/GO/homo_sapiens_proteome_GO_slim_boolean.parquet.gz')
# Convert to readable
df.columns = ['id', 'sequence'] + [go_to_readable(gs, godag) for gs in df.columns[2:]]
df

In [ ]:
import polars as po
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

def visualize_correlations(
    df: po.DataFrame,
    n: int = 3,
    method: str = "pearson"
):
    """
    Visualize top, bottom, and middle correlations from a Polars DataFrame
    using seaborn.clustermap (viridis colormap, square figure, labeled axes).

    Parameters
    ----------
    df : po.DataFrame
        Input Polars DataFrame.
    n : int
        Number of correlations to take from each group (top, middle, bottom).
    method : str
        Correlation method ("pearson", "spearman", "kendall").
    """
    # Keep only numeric columns
    numeric_cols = [c for c, dtype in zip(df.columns, df.dtypes) if dtype != po.Utf8]
    df_num = df[numeric_cols]

    if len(numeric_cols) < 2:
        raise ValueError("Need at least 2 numeric columns for correlations.")

    # Compute correlation matrix
    corr = df_num.to_pandas().corr(method=method)

    # Flatten correlation matrix (ignore self-corr and duplicates)
    corr_values = (
        corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .reset_index()
    )
    corr_values.columns = ["col1", "col2", "corr"]

    # If n*3 >= total pairs, just use all columns
    max_pairs = len(corr_values)
    if n * 3 >= max_pairs:
        selected_vars = corr.columns
        col_labels = {col: "all" for col in selected_vars}
    else:
        sorted_corr = corr_values.sort_values("corr").reset_index(drop=True)

        # Select unique columns from the top n pairs
        bottom_cols = sorted_corr.head(n)[['col1', 'col2']].values.flatten()
        bottom_cols = list(np.unique(bottom_cols))

        # Select unique columns from the middle n pairs
        middle_idx_start = max(0, len(sorted_corr) // 2 - n // 2)
        middle_cols = sorted_corr.iloc[middle_idx_start : middle_idx_start + n][['col1', 'col2']].values.flatten()
        middle_cols = list(np.unique(middle_cols))

        # Select unique columns from the bottom n pairs
        top_cols = sorted_corr.tail(n)[['col1', 'col2']].values.flatten()
        top_cols = list(np.unique(top_cols))

        # Combine and remove duplicates from the groups themselves
        selected_vars = list(set(bottom_cols + middle_cols + top_cols))

        # Create labels for coloring ticks
        col_labels = {}
        for col in bottom_cols:
            col_labels[col] = "bottom"
        for col in middle_cols:
            col_labels[col] = "middle"
        for col in top_cols:
            col_labels[col] = "top"

    # Subset correlation matrix
    sub_corr = corr.loc[selected_vars, selected_vars]

    # Determine square figure size proportional to number of columns
    n_cols = sub_corr.shape[0]
    fig_size = max(6, n_cols * 0.8)

    # Plot clustermap
    g = sns.clustermap(
        sub_corr,
        cmap="viridis",
        center=0,
        linewidths=0.5,
        figsize=(fig_size, fig_size),
        cbar_kws={"label": "Correlation"},
        xticklabels=True,
        yticklabels=True
    )

    # Rotate x-axis labels for readability
    plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    # Color x/y tick labels based on top/middle/bottom
    tick_colors = {"bottom": "blue", "middle": "gray", "top": "red", "all": "black"}
    for lbl in g.ax_heatmap.get_xticklabels():
        text = lbl.get_text()
        lbl.set_color(tick_colors.get(col_labels.get(text, "black")))
    for lbl in g.ax_heatmap.get_yticklabels():
        text = lbl.get_text()
        lbl.set_color(tick_colors.get(col_labels.get(text, "black")))

    plt.show()

# Example usage (assuming 'df' is defined)
# visualize_correlations(df, n=5)


In [ ]:
df.with_columns(
    po.col(c).cast(po.Float32) for c in df.columns if df[c].dtype in [po.Float64, po.Int64, po.Boolean]
)

In [ ]:
df2 = po.read_parquet('/home/mila/s/shawn.whitfield/scratch/data/tasks/prot_param/h_sapiens_proteome_prot_param_protein_level.parquet.gz')
df2

In [ ]:
df2.with_columns(
    po.col(c).cast(po.Float32) for c in df2.columns if df2[c].dtype in [po.Float64, po.Int64, po.Boolean]
)

In [ ]:
datamodule.data_path

In [ ]:
# from project.datamodules.datamodule import PLMDataModule
# datamodule = PLMDataModule(data_path='/home/mila/s/shawn.whitfield/scratch/data/tasks/GO/homo_sapiens_proteome_GO_slim_boolean.parquet.gz',
#                         task_level = 'protein_level',
#                         task_type = 'binary_classification',)

# Load from checkpoint
checkpoint_path = '/home/mila/s/shawn.whitfield/projects/plm_interpretability/project/slurm_runs/logs/linear_probing_facebook/esm2_t6_8M_UR50D_l6/checkpoints/last.ckpt'
from project.algorithms.lightning_module import ProbingLightningModule
model = ProbingLightningModule.load_from_checkpoint(checkpoint_path, datamodule=datamodule)
model

In [ ]:
state_dict = torch.load(checkpoint_path, map_location=DEVICE)
ProbingLightningModule.load_state_dict(state_dict = state_dict)

In [ ]:
from project.utils.strs import PROTEIN_LENGTH_CUTOFF
from project.utils.functions import read_fasta, one_hot_polars_column

df = po.read_csv('/home/mila/s/shawn.whitfield/scratch/data/datasets/h_sapiens_proteome/uniprotkb_AND_model_organism_9606_2025_09_09.tsv.gz', separator='\t').rename({'Entry':'id', 'Sequence': 'sequence'})
# Select only entries that match the GO annotated files we have
ids, _ = read_fasta('/home/mila/s/shawn.whitfield/scratch/data/datasets/h_sapiens_proteome/UP000005640_9606.fasta.gz', separator='|')
df = df.filter(po.col("id").is_in(ids))
# Remove proteins with length > length_cutoff
df = df.filter(po.col('sequence').str.len_chars() < PROTEIN_LENGTH_CUTOFF)

one_hot_interpro = one_hot_polars_column('InterPro', df, cols_to_keep=['id', 'sequence'], occurrence_threshold=10)
one_hot_interpro.write_parquet('/home/mila/s/shawn.whitfield/scratch/data/tasks/domains/homo_sapiens_proteome_one_hot_interpro_domains.parquet.gz', compression='gzip')

In [ ]:
visualize_correlations(one_hot_interpro)

In [ ]:
# df = po.read_parquet('/home/mila/s/shawn.whitfield/scratch/data/tasks/prot_param/h_sapiens_proteome_prot_param_protein_level.parquet.gz')
df = po.read_parquet('/home/mila/s/shawn.whitfield/scratch/data/tasks/GO/homo_sapiens_proteome_GO_slim_boolean.parquet.gz')
df

In [ ]:
check_correlation(df, cmap='viridis')

In [ ]:
PROTEIN_LENGTH_CUTOFF = 1024

In [ ]:
# Get and store protein language model embeddings for a set of sequences
import torch
import pickle
from tqdm import tqdm # For a progress bar
from pathlib import Path
import polars as po
from project.utils.functions import set_device
DEVICE = set_device()
from project.algorithms.networks.protein_language_model import ProteinLanguageModel
from project.datamodules.datamodule import PolarsDataset 
from project.utils.functions import read_fasta
from torch.utils.data import DataLoader


model_name = 'facebook/esm2_t6_8M_UR50D'

embedding_dir = Path(f'/home/mila/s/shawn.whitfield/scratch/data/embeddings/{model_name.replace('/', '_')}')
embedding_dir.mkdir(parents=True, exist_ok=True)
batch_size = 64


# Read the fasta - whole human proteome
data_path = Path('/home/mila/s/shawn.whitfield/scratch/data/datasets/h_sapiens_proteome')
dataset_name = 'h_sapiens_proteome'
all_ids, all_seqs = read_fasta(data_path / 'UP000005640_9606.fasta.gz', separator='|')
all_seqs = [str(s) for s in all_seqs] # Have to convert Seq() to str

# Read in the dataset
df = po.DataFrame({'id': all_ids, 'sequence': all_seqs})
# Remove proteins with length > length_cutoff
df = df.filter(po.col('sequence').str.len_chars() < PROTEIN_LENGTH_CUTOFF)

# Turn into a dataloader
dataloader = DataLoader(
            PolarsDataset(df),
            batch_size=batch_size,
            shuffle=False,
        )

# Make the model
model = ProteinLanguageModel(model_name, layer_to_use=None)

with torch.no_grad():
    for batch_num, batch in enumerate(tqdm(dataloader)):
    # ids = next(iter(dataloader))['id']
    # sequences = next(iter(dataloader))['sequence']

        ids = batch['id']
        sequences = batch['sequence']

        # Run sequences through the model
        hidden_layers, attention_masks = model(sequences)

        # Apply attention to each hidden layer to remove focus on amino acids, not padding tokens
        hidden_layers = [(hl*attention_masks.unsqueeze(-1)).detach().to('cpu').numpy() for hl in hidden_layers]
    
        # Make structure to save (simple dict)
        to_save = {'ids': ids, 'sequences': sequences, 'hidden_embeddings': hidden_layers, 'attention_masks': attention_masks.detach().to('cpu').numpy()}

        with open(embedding_dir / f"{dataset_name}_{model_name.replace('/', '_')}_batch_{batch_num}.pkl", 'wb') as f:
            pickle.dump(to_save, f)

In [ ]:
model_name = 'facebook/esm2_t12_35M_UR50D'
# Get all pickled files with 'batch' in the name, in the embedding directory
embedding_dir = Path(f'/home/mila/s/shawn.whitfield/scratch/data/embeddings/{model_name.replace('/', '_')}')
all_files = list(embedding_dir.glob('*batch*.pkl'))
'_'.join(all_files[0].name.split('_')[:-2])

In [ ]:
# with torch.no_grad():
#     ids = next(iter(dataloader))['id']
#     sequences = next(iter(dataloader))['sequence']

#     batch_num = 1
#     # Run sequences through the model
#     hidden_layers, attention_masks = model(sequences)
#     # Collect hiddens
#     hidden_layers = torch.stack(hidden_layers).detach().to('cpu').numpy() # (num_layers, batch_size, seq_len, hidden_size)
#     # Collect attentions
#     attention_masks = attention_masks.detach().to('cpu').numpy() # (num_layers, batch_size, seq_len, hidden_size)
#     # Make structure to save (simple dict)
#     to_save = {'ids': ids, 'sequences': sequences, 'hidden_embeddings': hidden_layers, 'attention_masks': attention_masks}

#     with open(embedding_dir / f"{dataset_name}_{model_name.replace('/', '_')}_batch_{batch_num}.pkl", 'wb') as f:
#         pickle.dump(to_save, f)

In [ ]:
torch.stack(hiddens).shape  # (num_layers, batch_size, seq_len, hidden_size)

In [ ]:
hidden_layers[4].shape

In [ ]:
# Get and store protein language model embeddings for a set of sequences
import torch
import pickle
from tqdm import tqdm # For a progress bar
from pathlib import Path
import polars as po
from project.utils.functions import set_device
DEVICE = set_device()
from project.algorithms.networks.protein_language_model import ProteinLanguageModel
from project.datamodules.datamodule import PolarsDataset 
from project.utils.functions import read_fasta
from torch.utils.data import DataLoader


model_name = 'facebook/esm2_t12_35M_UR50D'

embedding_dir = Path(f'/home/mila/s/shawn.whitfield/scratch/data/embeddings/{model_name.replace('/', '_')}')
embedding_dir.mkdir(parents=True, exist_ok=True)
batch_size = 36
length_cutoff = 2048


# Read the fasta - whole human proteome
data_path = Path('/home/mila/s/shawn.whitfield/scratch/data/datasets/h_sapiens_proteome')
dataset_name = 'h_sapiens_proteome'
all_ids, all_seqs = read_fasta(data_path / 'UP000005640_9606.fasta.gz', separator='|')
all_seqs = [str(s) for s in all_seqs]

# Read in the dataset
df = po.DataFrame({'id': all_ids, 'sequence': all_seqs})
# Remove proteins with length > length_cutoff
df = df.filter(po.col('sequence').str.len_chars() < length_cutoff)

# Turn into a dataloader
dataloader = DataLoader(
            PolarsDataset(df),
            batch_size=batch_size,
            shuffle=False,
        )

# Make the model
model = ProteinLanguageModel(model_name, layer_to_use=None)

with torch.no_grad():
    # for batch_num, batch in enumerate(tqdm(dataloader)):
    ids = next(iter(dataloader))['id']
    sequences = next(iter(dataloader))['sequence']

        # ids = batch['id']
        # sequences = batch['sequence']

    # Run sequences through the model
    hidden_layers, attention_masks = model(sequences)

In [ ]:
hiddens = [hl*attention_masks.unsqueeze(-1) for hl in hidden_layers]

In [ ]:
hiddens[0].detach().to('cpu').numpy().shape

In [ ]:
np.pad(attention_masks.detach().to('cpu').numpy(), (0,5)).shape

In [ ]:
import numpy as np
np.pad(hiddens[0].detach().to('cpu').numpy(), (2,0)).shape

In [ ]:
torch.nn.functional.pad(hiddens[0], (0,0,0,2), 'constant', 0.0).shape

In [ ]:
torch.nn.functional.pad(hiddens[0], (0,0,0,2), 'constant', 0)[2,:,2]

In [ ]:
embeddings.shape

In [ ]:
attention_masks.unsqueeze(-1).shape

In [ ]:
# Pass sequences through the model in batches of n
for i in range(0, batch_size, batch_size):
    batch_seqs = sequences[i:i+batch_size]
    outputs = model(batch_seqs)

In [ ]:
model()

In [ ]:


dataset_path = ???
model_name = ???
batch_size = 128




# Make the model
model, tokenizer = ProteinLanguageModel.load_plm(model_name)

# Pass sequences through the model in batches of n
for i in range(0, len(sequences), batch_size):
    batch_seqs = sequences[i:i+batch_size]
    # Tokenize the sequences and move to device
    hidden_states, attention_mask = model(batch_seqs)

In [ ]:

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from utils.functions import read_fasta, set_device

#### ML

In [ ]:
import torch

device = set_device()

#### Bioinformatics

In [ ]:
from goatools.anno.gaf_reader import GafReader

# Loading data and preprocessing

In [ ]:
import polars as po

In [ ]:
from pathlib import Path
data_path = Path.cwd().parent.parent.parent.parent / 'scratch' / 'data'
column_names = [
        "DB", "DB_Object_ID", "DB_Object_Symbol", "Qualifier", "GO_ID", 
        "DB_Reference", "Evidence_Code", "With_From", "Aspect", "DB_Object_Name",
        "DB_Object_Synonym", "DB_Object_Type", "Taxon", "Date", "Assigned_By",
        "Annotation_Extension", "Gene_Product_Form_ID"
    ]
df = po.scan_csv(data_path / 'goa_uniprot_all.gaf.gz', separator='\t', 
                 has_header=False, 
                 new_columns=column_names).select(["DB_Object_ID", "GO_ID", "Evidence_Code", "Aspect"]).collect()
df.write_parquet('goa_uniprot_gaf.parquet.gz', compression='gzip')

In [ ]:
from pathlib import Path
data_path = Path.cwd().parent.parent.parent.parent / 'scratch' / 'data'

def get_annotations(ids):
    from goatools.anno.gaf_reader import GafReader
    # Load gaf file into the gaf reader
    gaf = GafReader(data_path / 'goa_uniprot_all.gaf')
    # Annotations are stored in three dicts, one for each godag branch
    ns2assc = gaf.get_ns2assc()

    filtered_dict = {} # this will be a cut-down version 
    for namespace, associations in ns2assc.items():
        # We'll keep only annotations (id:GO_terms) that match our input ids
        filtered_annotations = {}
        for accession, go_terms in associations.items():
            if accession in ids:
                filtered_annotations[accession] = go_terms
        filtered_dict[namespace] = filtered_annotations

    return filtered_dict

# Test with small fasta
filepath = '/home/mila/s/shawn.whitfield/scratch/data/miniref50.fasta'
# Read fasta and get ids
all_ids, all_seqs = read_fasta(filepath, uniref=True) # Setting uniref = True gives the reference identifier that we can lookup in uniprot-kb
# Use the ids to get filtered annotations
filtered_ns2assc = get_annotations(all_ids)

In [ ]:
# Make a polars dataframe
data = {'sequence': all_seqs,
        'uniprot_kb_id': all_ids,
        'go_bp': [filtered_ns2assc['BP'][i] if i in all_ids else '' for i in  filtered_ns2assc['BP'].keys()],
        'go_mf': [filtered_ns2assc['MF'][i] if i in all_ids else '' for i in  filtered_ns2assc['MF'].keys()],
        'go_cc': [filtered_ns2assc['CC'][i] if i in all_ids else '' for i in  filtered_ns2assc['CC'].keys()],
        } # feed a dict into polars
df = po.DataFrame(data)

In [ ]:
data_path = Path.cwd().parent.parent.parent.parent / 'scratch' / 'data'
gaf = GafReader(data_path / 'goa_uniprot_all.gaf')
# Annotations are stored in three dicts, one for each godag branch
ns2assc = gaf.get_ns2assc()
ns2assc

In [ ]:

# folder_path = Path.cwd().parent / "data"
# # fasta_path = folder_path / "uniref50.fasta.gz"
# fasta_path = folder_path / "miniref50.fasta"
# annotations_path = folder_path / "goa_uniprot_all.gaf.gz"
# obo_path = folder_path / "go-basic.obo"

Protein ids

In [ ]:
records = read_fasta(fasta_path)
records

In [ ]:
# Map from uniref ID to UniprotKB ID


In [ ]:
# def parse_fasta(fasta_path):
#     """
#     Given a filepath, parses a Uniref FASTA file and returns a list of tuples: (protein ID, protein sequence).
#     """
#     it = 0
#     records = []
#     with gzip.open(fasta_path, "rt") as handle:
#         # Get a SeqRecord object for each record in the FASTA file
#         for record in SeqIO.parse(handle, "fasta"):
#             if it < 50:
#                 records.append(record)
#                 # We can extract the sequence and id
#                 # record_id = record.id.split("_")[1] # Ids start with "UniRef50_"
#                 # record_sequence = str(record.seq)
#                 # records.append((record_id, record_sequence))
#                 it += 1
#             else:
#                 break
#     return records

Protein ids to GO terms

In [ ]:
test_ids = [records.id.split("_")[1] for records in records]
test_ids

In [ ]:
ogaf = GafReader(annotations_path)

# Loading model and getting embeddings

In [ ]:
# Load model directly
from transformers import AutoModelForMaskedLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t30_150M_UR50D")
model = AutoModelForMaskedLM.from_pretrained("facebook/esm2_t30_150M_UR50D")

In [ ]:
type(model)

In [ ]:
test_seq = "MSASAVYVLDLKGKVLICRNYRGDVDMSEVEHFMPILMEKEEEGMLSPILAHGGVRFMWIKHNNLYLVATSKKNACVSLVFSFLYKVVQVFSEYFKELEEESIRDNFVIIYELLDELMDFGYPQTTDSKILQEYITQEGHKLETGAPRPPATVTNAVSWRSEGIKYRKNEVFLDVIESVNLLVSANGNVLRSEIVGSIKMRVFLSGMPELRLGLNDKVLFDNTGRGKSKSVELEDVKFHQCVRLSRFENDRTISFIPPDGEFELMSYRLNTHVKPLIWIESVIEKHSHSRIEYMIKAKSQFKRRSTANNVEIHIPVPNDADSPKFKTTVGSVKWVPENSEIVWSIKSFPGGKEYLMRAHFGLPSVEAEDKEGKPPISVKFEIPYFTTSGIQVRYLKIIEKSGYQALPWVRYITQNGDYQLRTQ"

# Tokenize the sequence
inputs = tokenizer(test_seq, return_tensors="pt")
input_ids = inputs['input_ids']
# Make a forward pass through the model
with torch.no_grad():
    outputs = model(input_ids, output_hidden_states=True)
    hidden_states = outputs.hidden_states

In [ ]:
len(hidden_states)